# Mitchell’s OmniMart Retail Data Platform
## Bronze Layer - Data Ingestion

### Project Overview
This notebook generates and ingests synthetic omnichannel retail data into the Bronze layer of a Microsoft Fabric Lakehouse architecture.

The project simulates a modern retail business operating through:
- Online sales channels
- Physical store transactions
- Product returns
- Inventory management

### Technologies Used
- Microsoft Fabric
- Lakehouse Architecture
- PySpark
- Delta Tables
- Faker (Synthetic Data Generation)

### Bronze Layer Objectives
- Generate raw retail datasets
- Simulate realistic business operations
- Store raw data in partitioned Delta tables
- Prepare data for Silver layer transformations

In [34]:
!pip install faker

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 36, Finished, Available, Finished, False)

## 1. Customer Data Generation

This section generates synthetic customer data using the Faker library.

Key features:
- GTA-based customer geography
- Customer segmentation
- Latitude and longitude generation
- Omnichannel retail customer simulation

Output:
- Raw CSV files
- bronze_customers Delta table

In [35]:
from faker import Faker
import pandas as pd
import random

fake = Faker("en_CA")

city_list = [
    "Toronto", "Mississauga", "Brampton", "Oakville", "Burlington",
    "Markham", "Vaughan", "Richmond Hill", "Pickering", "Ajax"
]

segments = ["New", "Regular", "VIP"]

customers = []

for i in range(200):
    customers.append({
        "customer_id": i + 1,
        "customer_name": fake.name(),
        "email": fake.email(),
        "address": fake.street_address(),
        "city": random.choice(city_list),
        "province": "Ontario",
        "country": "Canada",
        "postal_code": fake.postcode(),
        "customer_segment": random.choices(
            segments,
            weights=[30, 55, 15],
            k=1
        )[0],
        "signup_date": fake.date_between(start_date="-2y", end_date="today")
    })

df_customers = pd.DataFrame(customers)

display(df_customers.head())

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c29a8fe-5a78-4ae7-8393-82df226d4103)

In [36]:
df_spark_customers = spark.createDataFrame(df_customers)

df_spark_customers.write.mode("overwrite").option("header", "true").csv("Files/raw/customers/")

df_spark_customers.write.mode("overwrite").format("delta").saveAsTable("bronze_customers")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 38, Finished, Available, Finished, False)

In [37]:
df_spark_customers.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("bronze_customers")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 39, Finished, Available, Finished, False)

In [38]:
spark.sql("DESCRIBE bronze_customers").show(truncate=False)

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 40, Finished, Available, Finished, False)

+----------------+---------+-------+
|col_name        |data_type|comment|
+----------------+---------+-------+
|customer_id     |bigint   |NULL   |
|customer_name   |string   |NULL   |
|email           |string   |NULL   |
|address         |string   |NULL   |
|city            |string   |NULL   |
|province        |string   |NULL   |
|country         |string   |NULL   |
|postal_code     |string   |NULL   |
|customer_segment|string   |NULL   |
|signup_date     |date     |NULL   |
+----------------+---------+-------+



In [39]:
display(spark.sql("SELECT * FROM bronze_customers LIMIT 10"))

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5ac28da9-f987-45e3-92a4-fc739d36a86f)

In [40]:
spark.sql("""
SELECT 
    city,
    COUNT(*) as customers
FROM bronze_customers
GROUP BY city
""").show()

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 42, Finished, Available, Finished, False)

+-------------+---------+
|         city|customers|
+-------------+---------+
|    Pickering|       19|
|         Ajax|       14|
|Richmond Hill|       20|
|   Burlington|       20|
|      Toronto|       22|
|      Markham|       25|
|  Mississauga|       20|
|     Brampton|       21|
|     Oakville|       19|
|      Vaughan|       20|
+-------------+---------+



Location Mapping for Customers

In [41]:
gta_locations = {
    "Toronto": {"region": "Toronto Core", "lat": 43.6532, "lon": -79.3832},
    "Mississauga": {"region": "West GTA", "lat": 43.5890, "lon": -79.6441},
    "Brampton": {"region": "West GTA", "lat": 43.7315, "lon": -79.7624},
    "Oakville": {"region": "West GTA", "lat": 43.4675, "lon": -79.6877},
    "Burlington": {"region": "West GTA", "lat": 43.3255, "lon": -79.7990},
    "Markham": {"region": "North/East GTA", "lat": 43.8561, "lon": -79.3370},
    "Vaughan": {"region": "North/East GTA", "lat": 43.8372, "lon": -79.5083},
    "Richmond Hill": {"region": "North/East GTA", "lat": 43.8828, "lon": -79.4403},
    "Pickering": {"region": "North/East GTA", "lat": 43.8384, "lon": -79.0868},
    "Ajax": {"region": "North/East GTA", "lat": 43.8509, "lon": -79.0204}
}

locations = []

for i, (city, details) in enumerate(gta_locations.items()):
    locations.append({
        "location_id": i + 1,
        "city": city,
        "province": "Ontario",
        "country": "Canada",
        "gta_region": details["region"],
        "latitude": details["lat"],
        "longitude": details["lon"]
    })

df_locations = pd.DataFrame(locations)

display(df_locations)

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d2e01f7-415f-4128-88d8-861f57d6fcb7)

In [42]:
df_spark_locations = spark.createDataFrame(df_locations)

df_spark_locations.write.mode("overwrite").option("header", "true").csv("Files/raw/locations/")

df_spark_locations.write.mode("overwrite").format("delta").saveAsTable("bronze_locations")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 44, Finished, Available, Finished, False)

In [43]:
df_spark_locations.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("bronze_locations")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 45, Finished, Available, Finished, False)

#### 3. Product Data Generation

This section generates retail product data across multiple categories.

Business logic includes:
- Realistic product brands
- Retailer simulation
- Category and subcategory hierarchy
- Pricing and profit margin generation

Output:
- Raw CSV files
- bronze_products Delta table

In [44]:
import pandas as pd
import random

categories = ["Electronics", "Clothing", "Grocery", "Home", "Beauty"]

retailers = [
    "Walmart",
    "Amazon",
    "Costco",
    "Uber",
    "No Frills",
    "Loblaws",
    "Shoppers Drug Mart",
    "Sephora"
]

products = []

for product_id in range(1, 51):  # 50 products
    category = random.choice(categories)
    retailer = random.choice(retailers)

    if category == "Electronics":
        cost_price = round(random.uniform(80, 1200), 2)
        margin = random.uniform(0.12, 0.35)

    elif category == "Clothing":
        cost_price = round(random.uniform(8, 120), 2)
        margin = random.uniform(0.30, 0.65)

    elif category == "Grocery":
        cost_price = round(random.uniform(2, 40), 2)
        margin = random.uniform(0.10, 0.30)

    elif category == "Home":
        cost_price = round(random.uniform(15, 700), 2)
        margin = random.uniform(0.20, 0.50)

    else:  # Beauty
        cost_price = round(random.uniform(5, 150), 2)
        margin = random.uniform(0.35, 0.70)

    selling_price = round(cost_price * (1 + margin), 2)

    products.append({
        "product_id": product_id,
        "retailer_name": retailer,
        "category": category,
        "cost_price": cost_price,
        "selling_price": selling_price,
        "profit_margin": round(margin, 2),
        "is_active": random.choices([True, False], weights=[95, 5], k=1)[0]
    })

df_products = pd.DataFrame(products)

display(df_products.head())

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91760ab1-1743-4972-ad37-1ec1d22f8629)

In [45]:
df_spark_products = spark.createDataFrame(df_products)

df_spark_products.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("Files/raw/products/")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 47, Finished, Available, Finished, False)

In [46]:
df_spark_products.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("bronze_products")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 48, Finished, Available, Finished, False)

In [47]:
spark.sql("""
SELECT 
    category,
    COUNT(*) AS product_count,
    ROUND(AVG(selling_price), 2) AS avg_selling_price,
    ROUND(AVG(profit_margin), 2) AS avg_margin
FROM bronze_products
GROUP BY category
ORDER BY category
""").show()

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 49, Finished, Available, Finished, False)

+-----------+-------------+-----------------+----------+
|   category|product_count|avg_selling_price|avg_margin|
+-----------+-------------+-----------------+----------+
|     Beauty|            6|           100.24|      0.52|
|   Clothing|           14|            95.55|      0.46|
|Electronics|           11|           868.61|      0.24|
|    Grocery|           10|            28.77|      0.21|
|       Home|            9|           547.94|      0.39|
+-----------+-------------+-----------------+----------+



## 3. Orders Data Generation

This section generates customer order transactions for the 2025 retail year.

Features:
- Monthly order volume simulation
- Multiple sales channels
- Payment method generation
- Order status tracking

Partition Strategy:
- order_year
- order_month

Output:
- Raw partitioned files
- bronze_orders Delta table

In [48]:
from datetime import datetime, timedelta
import pandas as pd
import random

orders = []

sales_channels = ["Online", "In-Store"]
payment_methods = ["Credit Card", "Debit Card", "PayPal", "Apple Pay", "Gift Card"]

order_id = 1

for month in range(1, 13):
    month_start = datetime(2025, month, 1)

    if month == 12:
        next_month_start = datetime(2026, 1, 1)
    else:
        next_month_start = datetime(2025, month + 1, 1)

    days_in_month = (next_month_start - month_start).days
    monthly_order_count = random.randint(1000, 2500)

    for i in range(monthly_order_count):
        order_date = month_start + timedelta(
            days=random.randint(0, days_in_month - 1),
            hours=random.randint(8, 22),
            minutes=random.randint(0, 59)
        )

        orders.append({
            "order_id": order_id,
            "customer_id": random.randint(1, 200),
            "order_date": order_date,
            "sales_channel": random.choices(
                sales_channels,
                weights=[65, 35],
                k=1
            )[0],
            "payment_method": random.choice(payment_methods),
            "order_status": random.choices(
                ["Completed", "Returned", "Cancelled"],
                weights=[88, 8, 4],
                k=1
            )[0],
            "order_year": order_date.year,
            "order_month": order_date.month
        })

        order_id += 1

df_orders = pd.DataFrame(orders)

display(df_orders.head())

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ebf45818-e66f-4fa3-a976-21eae3a377c9)

In [49]:
df_spark_orders = spark.createDataFrame(df_orders)

df_spark_orders.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("order_year", "order_month") \
    .csv("Files/raw/orders/")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 51, Finished, Available, Finished, False)

In [50]:
df_spark_orders.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .format("delta") \
    .saveAsTable("bronze_orders")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 52, Finished, Available, Finished, False)

In [51]:
spark.sql("""
SELECT 
    order_year,
    order_month,
    COUNT(*) AS total_orders
FROM bronze_orders
GROUP BY order_year, order_month
ORDER BY order_year, order_month
""").show(12)

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 53, Finished, Available, Finished, False)

+----------+-----------+------------+
|order_year|order_month|total_orders|
+----------+-----------+------------+
|      2025|          1|        1259|
|      2025|          2|        1717|
|      2025|          3|        1205|
|      2025|          4|        2008|
|      2025|          5|        1446|
|      2025|          6|        2164|
|      2025|          7|        1252|
|      2025|          8|        2156|
|      2025|          9|        1959|
|      2025|         10|        1056|
|      2025|         11|        1646|
|      2025|         12|        2384|
+----------+-----------+------------+



## 4. Order Items Fact Generation

This section creates the primary transactional fact table.

Each order may contain multiple products, allowing:
- Basket analysis
- Revenue calculations
- Profit analysis
- Discount simulation

Business metrics generated:
- Revenue
- Profit
- Discounts
- Quantity sold

Partition Strategy:
- order_year
- order_month

Output:
- Raw partitioned files
- bronze_order_items Delta table

In [52]:
orders_pd = spark.table("bronze_orders").toPandas()
products_pd = spark.table("bronze_products").select(
    "product_id",
    "selling_price",
    "cost_price"
).toPandas()

product_records = products_pd.to_dict("records")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 54, Finished, Available, Finished, False)

In [53]:
order_items = []
order_item_id = 1

for _, order in orders_pd.iterrows():
    items_in_order = random.randint(1, 5)
    selected_products = random.sample(product_records, items_in_order)

    for product in selected_products:
        quantity = random.choices(
            [1, 2, 3, 4],
            weights=[65, 22, 9, 4],
            k=1
        )[0]

        unit_price = float(product["selling_price"])
        unit_cost = float(product["cost_price"])

        discount_rate = random.choices(
            [0, 0.05, 0.10, 0.15, 0.20],
            weights=[70, 12, 10, 5, 3],
            k=1
        )[0]

        gross_amount = round(unit_price * quantity, 2)
        discount_amount = round(gross_amount * discount_rate, 2)
        line_total = round(gross_amount - discount_amount, 2)
        line_cost = round(unit_cost * quantity, 2)
        line_profit = round(line_total - line_cost, 2)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": int(order["order_id"]),
            "product_id": int(product["product_id"]),
            "quantity": quantity,
            "unit_price": unit_price,
            "unit_cost": unit_cost,
            "discount_rate": discount_rate,
            "discount_amount": discount_amount,
            "line_total": line_total,
            "line_cost": line_cost,
            "line_profit": line_profit,
            "order_year": int(order["order_year"]),
            "order_month": int(order["order_month"])
        })

        order_item_id += 1

df_order_items = pd.DataFrame(order_items)

display(df_order_items.head())

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d90e1bce-49fa-463c-9331-a0997f206fe8)

In [54]:
df_spark_order_items = spark.createDataFrame(df_order_items)

df_spark_order_items.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("order_year", "order_month") \
    .csv("Files/raw/order_items/")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 56, Finished, Available, Finished, False)

In [55]:
df_spark_order_items.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .format("delta") \
    .saveAsTable("bronze_order_items")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 57, Finished, Available, Finished, False)

In [56]:
spark.sql("""
SELECT 
    order_year,
    order_month,
    COUNT(*) AS total_order_items,
    ROUND(SUM(line_total), 2) AS total_revenue,
    ROUND(SUM(line_profit), 2) AS total_profit
FROM bronze_order_items
GROUP BY order_year, order_month
ORDER BY order_year, order_month
""").show(12)

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 58, Finished, Available, Finished, False)

+----------+-----------+-----------------+-------------+------------+
|order_year|order_month|total_order_items|total_revenue|total_profit|
+----------+-----------+-----------------+-------------+------------+
|      2025|          1|             3769|    1942276.0|   415213.37|
|      2025|          2|             5039|   2470607.95|   523388.31|
|      2025|          3|             3658|   1796614.26|   380501.67|
|      2025|          4|             5971|   3059122.16|   639250.41|
|      2025|          5|             4345|   2036275.85|   436697.17|
|      2025|          6|             6369|   3068186.49|    647250.3|
|      2025|          7|             3734|   1887376.49|   397511.45|
|      2025|          8|             6445|   3129584.43|   665418.48|
|      2025|          9|             5774|   2791603.63|   589014.16|
|      2025|         10|             3254|    1682140.3|   353189.33|
|      2025|         11|             4883|   2400443.27|   500666.85|
|      2025|        

In [57]:
spark.sql("""
SELECT 
    ROUND(COUNT(*) / COUNT(DISTINCT order_id), 2) AS avg_items_per_order
FROM bronze_order_items
""").show()

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 59, Finished, Available, Finished, False)

+-------------------+
|avg_items_per_order|
+-------------------+
|               2.98|
+-------------------+



## 5. Returns Data Generation

This section simulates customer product returns.

Returns are generated from completed order transactions and include:
- Refund amounts
- Return reasons
- Return dates

Returns are modeled separately from sales transactions to preserve historical sales records and support net revenue analysis.

Output:
- Raw partitioned files
- bronze_returns Delta table

In [58]:
orders_pd = spark.table("bronze_orders").toPandas()
order_items_pd = spark.table("bronze_order_items").toPandas()

returned_orders = orders_pd[orders_pd["order_status"] == "Returned"]

returns = []
return_id = 1

return_reasons = [
    "Damaged item",
    "Wrong item delivered",
    "Customer changed mind",
    "Size issue",
    "Late delivery",
    "Product not as described"
]

for _, order in returned_orders.iterrows():
    matching_items = order_items_pd[order_items_pd["order_id"] == order["order_id"]]

    if len(matching_items) > 0:
        returned_item = matching_items.sample(1).iloc[0]

        return_date = pd.to_datetime(order["order_date"]) + pd.Timedelta(days=random.randint(1, 21))

        returns.append({
            "return_id": return_id,
            "order_id": int(order["order_id"]),
            "order_item_id": int(returned_item["order_item_id"]),
            "product_id": int(returned_item["product_id"]),
            "customer_id": int(order["customer_id"]),
            "return_date": return_date,
            "return_reason": random.choice(return_reasons),
            "refund_amount": float(returned_item["line_total"]),
            "return_year": return_date.year,
            "return_month": return_date.month
        })

        return_id += 1

df_returns = pd.DataFrame(returns)

display(df_returns.head())

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 861bb0b5-9bda-4159-bd4f-d22c167ab3f6)

In [59]:
df_spark_returns = spark.createDataFrame(df_returns)

df_spark_returns.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("return_year", "return_month") \
    .csv("Files/raw/returns/")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 61, Finished, Available, Finished, False)

In [60]:
df_spark_returns.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("return_year", "return_month") \
    .format("delta") \
    .saveAsTable("bronze_returns")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 62, Finished, Available, Finished, False)

In [61]:
spark.sql("""
SELECT 
    return_year,
    return_month,
    COUNT(*) AS total_returns,
    ROUND(SUM(refund_amount), 2) AS total_refunds
FROM bronze_returns
GROUP BY return_year, return_month
ORDER BY return_year, return_month
""").show(20)

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 63, Finished, Available, Finished, False)

+-----------+------------+-------------+-------------+
|return_year|return_month|total_returns|total_refunds|
+-----------+------------+-------------+-------------+
|       2025|           1|           78|     32522.99|
|       2025|           2|          141|     67861.94|
|       2025|           3|          128|     77930.46|
|       2025|           4|          150|     75778.73|
|       2025|           5|          141|     63504.27|
|       2025|           6|          146|     70069.19|
|       2025|           7|          129|     65727.26|
|       2025|           8|          133|      73222.4|
|       2025|           9|          140|     51981.48|
|       2025|          10|          113|     56054.56|
|       2025|          11|          112|     68102.66|
|       2025|          12|          181|     76022.42|
|       2026|           1|           68|     26642.59|
+-----------+------------+-------------+-------------+



## 6. Inventory Snapshot Generation

This section generates monthly inventory snapshot data across warehouse locations.

Inventory logic includes:
- Beginning stock
- Units received
- Units sold
- Ending stock
- Stock status classification

Output:
- Raw partitioned files
- bronze_inventory Delta table

In [62]:
products_pd = spark.table("bronze_products").select(
    "product_id"
).toPandas()

inventory = []
inventory_id = 1

warehouse_locations = [
    "Toronto DC",
    "Mississauga DC",
    "Brampton DC",
    "Markham DC"
]

for month in range(1, 13):
    snapshot_date = pd.Timestamp(year=2025, month=month, day=1)

    for _, product in products_pd.iterrows():

        units_received = random.randint(10, 50)
        units_sold = random.randint(15, 50)

        ending_stock = max(
            random.randint(20, 100)
            + units_received
            - units_sold,
            0
        )

        inventory.append({
            "inventory_id": inventory_id,
            "snapshot_date": snapshot_date,
            "product_id": int(product["product_id"]),
            "warehouse_location": random.choice(warehouse_locations),
            "units_received": units_received,
            "units_sold": units_sold,
            "ending_stock": ending_stock,
            "stock_status": (
                "Out of Stock" if ending_stock == 0
                else "Low Stock" if ending_stock < 20
                else "In Stock"
            ),
            "snapshot_year": snapshot_date.year,
            "snapshot_month": snapshot_date.month
        })

        inventory_id += 1

df_inventory = pd.DataFrame(inventory)

display(df_inventory.head())

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4038cec1-4dec-4354-9d63-32d9ac9cbfb4)

In [63]:
df_spark_inventory = spark.createDataFrame(df_inventory)

df_spark_inventory.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("snapshot_year", "snapshot_month") \
    .csv("Files/raw/inventory/")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 65, Finished, Available, Finished, False)

In [64]:
df_spark_inventory.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("snapshot_year", "snapshot_month") \
    .format("delta") \
    .saveAsTable("bronze_inventory")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 66, Finished, Available, Finished, False)

In [65]:
spark.sql("""
SELECT 
    snapshot_year,
    snapshot_month,
    COUNT(*) AS inventory_records,
    SUM(ending_stock) AS total_ending_stock
FROM bronze_inventory
GROUP BY snapshot_year, snapshot_month
ORDER BY snapshot_year, snapshot_month
""").show(12)

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 67, Finished, Available, Finished, False)

+-------------+--------------+-----------------+------------------+
|snapshot_year|snapshot_month|inventory_records|total_ending_stock|
+-------------+--------------+-----------------+------------------+
|         2025|             1|               50|              3246|
|         2025|             2|               50|              2828|
|         2025|             3|               50|              2863|
|         2025|             4|               50|              2983|
|         2025|             5|               50|              3096|
|         2025|             6|               50|              2908|
|         2025|             7|               50|              2775|
|         2025|             8|               50|              2802|
|         2025|             9|               50|              2770|
|         2025|            10|               50|              3014|
|         2025|            11|               50|              2831|
|         2025|            12|               50|

# Bronze Layer Summary

The Bronze layer for Mitchell’s OmniMart Retail Data Platform has been successfully created and validated.

Datasets generated:
- Customers
- Products
- Orders
- Order Items
- Returns
- Inventory Snapshots

Key implementation features:
- Synthetic data generation using Faker
- Delta Lake storage format
- Partitioned transactional datasets
- Realistic retail business logic
- Omnichannel sales simulation
- Revenue, profit, and refund calculations
- Inventory and warehouse tracking

The generated datasets have been stored in:
- Raw file storage within the Lakehouse
- Bronze Delta tables for downstream processing

Next Steps:
The next phase of the project will focus on Silver layer transformations, including:
- Data cleaning
- Standardization
- Deduplication
- Business rule enforcement
- Star schema preparation

In [66]:
tables = [
    "bronze_customers",
    "bronze_products",
    "bronze_orders",
    "bronze_order_items",
    "bronze_returns",
    "bronze_inventory",
    "bronze_locations"
]

for table in tables:
    count = spark.table(table).count()
    print(f"{table}: {count:,} rows")

StatementMeta(, 9483dbb9-46e1-466d-98ca-57a8f26d6fbc, 68, Finished, Available, Finished, False)

bronze_customers: 200 rows
bronze_products: 50 rows
bronze_orders: 20,252 rows
bronze_order_items: 60,274 rows
bronze_returns: 1,660 rows
bronze_inventory: 600 rows
bronze_locations: 10 rows
